In [6]:
import os
import geopandas as gpd
import numpy as np
import rasterio

In [7]:
def clip_images(img_path,boundary,out_dir, grid = 50):

    ## This function creates 50 x 50 grids and clips the raster image into those chunks

    os.makedirs(out_dir,exist_ok = True) # create output directory
    bounds = gpd.read_file(boundary) # read the boundary file
    
    with rasterio.open(img_path) as img: # Opening the raster file
        bounds = bounds.to_crs(img.crs)  # reprojecting the boundary crs to match the raster crs
        img_bounds = img.bounds   # extracted the bounds of the raster
        img_crs = img.crs         # extracted Crs of the raster file
        if img.nodata is not None:
            nodata = img.nodata    # create a nodata variable 
        else:
            nodata = 0
        
        xmin, ymin, xmax, ymax = img_bounds  # assigned xmin, ymin, xmax, ymax to the image bounds
        
        grid_cells = [                                   # create a list for the 50 by 50 grids
            box(x, y, x + grid, y + grid)
            for x in np.arange(xmin, xmax, grid)
            for y in np.arange(ymin, ymax, grid)
        ]                                            

        # create a geodataframe from the grids
        grid_gdf = gpd.GeoDataFrame({'geometry': grid_cells}, crs=img_crs)  # created a geodataframe to store the geometries of the grids

        base = os.path.splitext(os.path.basename(img_path))[0]  # Extracted the filename of the raster file

        saved = 0
        
        ## Clip the raster file to the grids
        for idx, row in  grid_gdf.iterrows():

            b_mask = row['geometry']

            if not bounds.intersects(b_mask).any():
                continue
                
            out_image, out_transform = mask(img, [mapping(b_mask)],crop =True, nodata=nodata)

            is_nodata = np.all(out_image == nodata, axis = 0)
            is_white = np.all(out_image == nodata, axis = 0)
            empty_pct = np.mean(is_nodata | is_white) * 100

            if empty_pct >= 98:
                continue

            profile = img.meta.copy()
            profile.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
            })

            out_path = os.path.join(out_dir,f"{base}_tile_{idx}.tif")   ## Save clipped raster
            with rasterio.open(out_path,"w",**profile) as dst:
                dst.write(out_image)
            saved += 1

        print(f"Saved {saved} of {len(grid_gdf)} tiles")

def geo_to_pixel_coordinates(geojson_file, out_dir):

    ## This function is to convert geographic coordinates into pixel coordinates for model fine-tuning

    directory = os.path.split(out_dir)[0]               ## Create output directory
    path = os.path.join(directory, "Pixel_coordinates")
    os.makedirs(path, exist_ok = True)

    gdf = gpd.read_file(geojson_file)  # Read geojson or point file and save as a geodataframe
    
    for file in os.listdir(out_dir):
        input_path = os.path.join(out_dir, file)

        with rasterio.open(input_path) as img:          ## Opens clipped raster files
            base = os.path.splitext(os.path.basename(input_path))[0]
         
            transform = img.transform
            img_bounds = img.bounds   ## Extracts bounds of the image
            
            clip_gdf = gdf.to_crs(img.crs)        ## Reprojects geodataframe's CRS to match the CRS of the image
        
            clip_gdf = clip_gdf.clip(img_bounds)   ### Clips geodaframe to image bounds

            rows = []  # Empty list to store pixel coordinates (X, longitudes)
            cols = []  # Empty list to store pixel coordinates (Y, latitudes)
            for geom in clip_gdf.geometry:
                # print(geom)
                
                row, col = rasterio.transform.rowcol(transform, geom.x, geom.y)  # transforming geographic coordinates to pixel coordinates
                print(f"Pixel Row (Y): {row}, Pixel Column (X): {col}")
                rows.append(col)     # Append transformed values
                cols.append(row)

            df = pd.DataFrame({"X": rows, "Y" : cols})  # Create a dataframe to export coordinates in CSV format
            csv_path = os.path.join(path,f"{base}_pixel_coordinates.csv")
            df.to_csv(csv_path)
            print(f"Saved {base} pixel cordinates to {csv_path}")

def bounding_box_to_yolo(geojson_file,out_dir, class_id = 0):

## This function extracts crown diameter values, creates bounding boxes, and converts those bounding boxes into YOLO format required for model training
    
    directory = os.path.split(out_dir)[0]
    path = os.path.join(directory, "Yolo_format")  
    os.makedirs(path, exist_ok = True)      ## Create output directory

    gdf = gpd.read_file(geojson_file)       # Read geojson or point file and save as a geodataframe
    
    for file in os.listdir(out_dir):
        input_path = os.path.join(out_dir, file)

        with rasterio.open(input_path) as img:       
            base = os.path.splitext(os.path.basename(input_path))[0]
         
            inv = ~img.transform     ## Inverts affine transform to convert map coordinates to pixels
            W,H = img.width, img.height   ## Extract width and height
            tile = box(*img.bounds)       ## Unpacks the coordinates of the image and creates a rectangle polygon
            clip_gdf = gdf.to_crs(img.crs)  ## Reprojects geodataframe's CRS to match the CRS of the image

            boxes = clip_gdf.geometry.buffer(clip_gdf["Diameter"] / 2, cap_style=3)    ## Creates boxes using the diameter column of the geojson file
            boxes = boxes[boxes.intersects(tile)]    

        lines =[]  ## Create an empty list to store the YOLO formatted text line per box
        
        ## 
        if len(boxes):
            b = boxes.bounds
            col_min = inv.a * b.minx + inv.b * b.maxy + inv.c
            row_min = inv.d * b.minx + inv.e * b.maxy + inv.f
            col_max = inv.a * b.maxx + inv.b * b.miny + inv.c
            row_max = inv.d * b.maxx + inv.e * b.miny + inv.f

            col_min, col_max = np.clip(col_min, 0, W), np.clip(col_max, 0, W)
            row_min, row_max = np.clip(row_min, 0, H), np.clip(row_max, 0, H)
            keep = (col_max > col_min) & (row_max > row_min)

            xc = ((col_min + col_max)/ 2 / W)[keep]
            yc = ((row_min + row_max)/ 2 / H)[keep]
            bw = ((col_max - col_min)/ W)[keep]
            bh = ((row_max - row_min)/ H)[keep]

            lines = [f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
                    for x, y, w, h in zip(xc,yc,bw,bh)]

        label_path = os.path.join(path, base + ".txt")
        with open(label_path,"w") as f:
            f.write("\n".join(lines))

        print(f"{file}: {len(lines)} boxes")

In [ ]:
imagery = r""
boundary = r""
out_dir = r""
geojson_file = r""

clip_images(imagery,boundary,out_dir)
geo_to_pixel_coordinates(geojson_file, out_dir)
bounding_box_to_yolo(geojson_file,out_dir, class_id = 0)